# Advanced MoE Story Generator - GPU Training

This notebook trains an improved Mixture of Experts model on your powerful GPU server.

**Model Improvements:**
- 800M total parameters (150M active)
- 16 experts (vs 8 in base)
- 24 layers (vs 16 in base)
- 1280 hidden size (vs 1024 in base)
- SwiGLU activation (better than GELU)
- RMSNorm (faster than LayerNorm)
- All layers use MoE (not every other)

**Hardware Requirements:**
- 40GB+ VRAM GPU (A100, H100, or similar)
- Multi-GPU support enabled

## 1. GPU Detection and Setup

In [ ]:
import torch
import os
import sys

# Add project to path
sys.path.insert(0, os.path.abspath('.'))

print("=" * 60)
print("GPU DETECTION AND CONFIGURATION")
print("=" * 60)

# Check CUDA availability
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    
    # List all available GPUs
    print("\nAvailable GPUs:")
    for i in range(torch.cuda.device_count()):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_memory = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {gpu_name}")
        print(f"    Total Memory: {gpu_memory:.2f} GB")
        
        # Check current memory usage
        torch.cuda.set_device(i)
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved = torch.cuda.memory_reserved(i) / 1e9
        print(f"    Allocated: {allocated:.2f} GB")
        print(f"    Reserved: {reserved:.2f} GB")
        print(f"    Free: {gpu_memory - reserved:.2f} GB")
    
    # Select primary GPU (usually GPU 0)
    primary_gpu = 0
    torch.cuda.set_device(primary_gpu)
    print(f"\n✓ Primary GPU set to: GPU {primary_gpu}")
    print(f"  Device: {torch.cuda.get_device_name(primary_gpu)}")
    
    # Enable TF32 for better performance on Ampere+ GPUs (A100, H100)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("\n✓ TF32 enabled for Ampere+ GPUs")
    
else:
    print("\n⚠ WARNING: No CUDA GPUs detected!")
    print("Training will be EXTREMELY slow on CPU.")
    print("Please ensure CUDA is properly installed.")

print("\n" + "=" * 60)

## 2. Import Dependencies

In [ ]:
import yaml
from pathlib import Path
from transformers import PreTrainedTokenizerFast

from storyteller.model import StorytellerModel, ModelConfig
from storyteller.data.dataset import StoryDatasetPreloaded, get_dataloader
from storyteller.training.trainer import Trainer

print("✓ All dependencies imported successfully")

## 3. Load Configuration

In [ ]:
# Load the advanced MoE configuration
config_path = "configs/moe_advanced.yaml"

print(f"Loading configuration from: {config_path}")
with open(config_path, 'r') as f:
    config_dict = yaml.safe_load(f)

print("\nModel Configuration:")
print(f"  Hidden size: {config_dict['model']['hidden_size']}")
print(f"  Num layers: {config_dict['model']['num_layers']}")
print(f"  Num heads: {config_dict['model']['num_attention_heads']}")
print(f"  Num experts: {config_dict['model']['num_experts']}")
print(f"  Top-K experts: {config_dict['model']['top_k_experts']}")
print(f"  MoE frequency: {config_dict['model']['moe_frequency']}")
print(f"  Activation: {config_dict['model']['activation']}")
print(f"  Norm type: {config_dict['model']['norm_type']}")

print("\nTraining Configuration:")
print(f"  Batch size: {config_dict['training']['batch_size']}")
print(f"  Gradient accumulation: {config_dict['training']['gradient_accumulation_steps']}")
print(f"  Effective batch size: {config_dict['training']['batch_size'] * config_dict['training']['gradient_accumulation_steps']}")
print(f"  Learning rate: {config_dict['training']['learning_rate']}")
print(f"  Num epochs: {config_dict['training']['num_epochs']}")
print(f"  AMP dtype: {config_dict['training']['amp_dtype']}")

print("\n✓ Configuration loaded successfully")

## 4. Load Tokenizer and Data

In [ ]:
# Load tokenizer
tokenizer_path = config_dict['training']['tokenizer_path']
print(f"Loading tokenizer from: {tokenizer_path}")

tokenizer = PreTrainedTokenizerFast.from_pretrained(tokenizer_path)
print(f"✓ Tokenizer loaded")
print(f"  Vocabulary size: {len(tokenizer):,}")
print(f"  PAD token: {tokenizer.pad_token}")
print(f"  EOS token: {tokenizer.eos_token}")

# Update vocab size in config
config_dict['model']['vocab_size'] = len(tokenizer)

# Load datasets with caching
print("\nLoading datasets...")
train_dataset = StoryDatasetPreloaded(
    data_path=config_dict['training']['train_data_path'],
    tokenizer=tokenizer,
    max_seq_length=config_dict['model']['max_seq_length'],
    cache_dir=config_dict['training']['cache_dir'],
)

val_dataset = StoryDatasetPreloaded(
    data_path=config_dict['training']['val_data_path'],
    tokenizer=tokenizer,
    max_seq_length=config_dict['model']['max_seq_length'],
    cache_dir=config_dict['training']['cache_dir'],
)

print(f"✓ Datasets loaded")
print(f"  Train examples: {len(train_dataset):,}")
print(f"  Val examples: {len(val_dataset):,}")

## 5. Create Data Loaders

In [ ]:
# Create data loaders
train_dataloader = get_dataloader(
    train_dataset,
    batch_size=config_dict['training']['batch_size'],
    shuffle=True,
    num_workers=config_dict['training']['num_workers'],
    pin_memory=config_dict['training']['pin_memory'],
)

val_dataloader = get_dataloader(
    val_dataset,
    batch_size=config_dict['training']['batch_size'],
    shuffle=False,
    num_workers=config_dict['training']['num_workers'],
    pin_memory=config_dict['training']['pin_memory'],
)

print("✓ Data loaders created")
print(f"  Train batches: {len(train_dataloader):,}")
print(f"  Val batches: {len(val_dataloader):,}")

## 6. Create Model

In [ ]:
# Create model configuration
model_cfg = config_dict['model'].copy()
model_cfg.pop('config_name', None)

model_config = ModelConfig(**model_cfg)

print("Creating model...")
model = StorytellerModel(model_config)

# Calculate parameter counts
total_params = model_config.num_parameters() / 1e6
active_params = model_config.active_parameters() / 1e6

print("\n✓ Model created successfully")
print(f"  Total parameters: {total_params:.1f}M")
print(f"  Active parameters: {active_params:.1f}M")
print(f"  Sparsity ratio: {(1 - active_params/total_params)*100:.1f}%")
print(f"  MoE layers: {sum(1 for i in range(model_config.num_layers) if i % model_config.moe_frequency == 0)}")
print(f"  Dense layers: {sum(1 for i in range(model_config.num_layers) if i % model_config.moe_frequency != 0)}")

# Move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"\n✓ Model moved to: {device}")

## 7. Create Optimizer and Scheduler

In [ ]:
# Separate parameters for weight decay
decay_params = []
no_decay_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if 'bias' in name or 'ln' in name or 'norm' in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer_grouped_parameters = [
    {
        'params': decay_params,
        'weight_decay': config_dict['training']['weight_decay'],
    },
    {
        'params': no_decay_params,
        'weight_decay': 0.0,
    },
]

optimizer = torch.optim.AdamW(
    optimizer_grouped_parameters,
    lr=config_dict['training']['learning_rate'],
    betas=(0.9, 0.95),
    eps=1e-8,
)

print("✓ Optimizer created (AdamW)")
print(f"  Parameters with decay: {len(decay_params):,}")
print(f"  Parameters without decay: {len(no_decay_params):,}")

# Create learning rate scheduler
num_training_steps = (
    len(train_dataloader) // config_dict['training']['gradient_accumulation_steps']
    * config_dict['training']['num_epochs']
)
warmup_steps = config_dict['training']['warmup_steps']

from torch.optim.lr_scheduler import LambdaLR
import math

def lr_lambda(current_step: int):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    else:
        progress = float(current_step - warmup_steps) / float(
            max(1, num_training_steps - warmup_steps)
        )
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
        return 0.1 + (1.0 - 0.1) * cosine_decay

scheduler = LambdaLR(optimizer, lr_lambda)

print("✓ Scheduler created (Cosine with warmup)")
print(f"  Total training steps: {num_training_steps:,}")
print(f"  Warmup steps: {warmup_steps:,}")

## 8. Create Trainer

In [ ]:
# Get evaluation config
eval_config = config_dict['training'].get('evaluation', {})

# Create trainer
trainer = Trainer(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    use_amp=config_dict['training']['use_amp'],
    amp_dtype=config_dict['training']['amp_dtype'],
    gradient_accumulation_steps=config_dict['training']['gradient_accumulation_steps'],
    max_grad_norm=config_dict['training']['max_grad_norm'],
    save_dir=config_dict['training']['save_dir'],
    save_every_n_steps=config_dict['training']['save_every_n_steps'],
    eval_every_n_steps=config_dict['training']['eval_every_n_steps'],
    log_every_n_steps=config_dict['training']['log_every_n_steps'],
    keep_last_n_checkpoints=config_dict['training']['keep_last_n_checkpoints'],
    use_mlflow=config_dict['training']['use_mlflow'],
    mlflow_experiment_name=config_dict['training']['mlflow_experiment_name'],
    mlflow_run_name=config_dict['training']['mlflow_run_name'],
    mlflow_tracking_uri=config_dict['training'].get('mlflow_tracking_uri'),
    mlflow_log_system_metrics=config_dict['training'].get('mlflow_log_system_metrics', True),
    tokenizer=tokenizer,
    num_eval_samples=eval_config.get('num_eval_samples', 50),
    eval_max_length=eval_config.get('eval_max_length', 512),
    eval_temperature=eval_config.get('eval_temperature', 1.0),
    eval_top_k=eval_config.get('eval_top_k', 50),
    eval_top_p=eval_config.get('eval_top_p', 0.95),
)

print("✓ Trainer created successfully")
print(f"  Device: {device}")
print(f"  Mixed precision: {config_dict['training']['use_amp']}")
print(f"  AMP dtype: {config_dict['training']['amp_dtype']}")
print(f"  Gradient accumulation: {config_dict['training']['gradient_accumulation_steps']}")
print(f"  Effective batch size: {config_dict['training']['batch_size'] * config_dict['training']['gradient_accumulation_steps']}")

## 9. Start Training

**IMPORTANT:** This will take 2-3 days on a high-end GPU. Monitor GPU usage to ensure it's being utilized.

In [ ]:
# Optional: Start MLflow UI in a separate terminal
# Run: mlflow ui --port 8080
# Then visit: http://localhost:8080

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"\nModel: Advanced MoE ({total_params:.1f}M total, {active_params:.1f}M active)")
print(f"Epochs: {config_dict['training']['num_epochs']}")
print(f"Estimated time: 2-3 days on A100/H100")
print("\nMonitor GPU usage with: nvidia-smi -l 1")
print("Monitor training in MLflow UI: http://localhost:8080")
print("\n" + "=" * 60 + "\n")

# Start training
trainer.train(num_epochs=config_dict['training']['num_epochs'])

## 10. Monitor GPU Usage During Training

Run this cell in a separate notebook or terminal to monitor GPU usage:

In [ ]:
# GPU monitoring (run this while training is happening)
import time

def monitor_gpu(duration_seconds=60, interval=5):
    """Monitor GPU usage for a specified duration."""
    if not torch.cuda.is_available():
        print("No CUDA GPUs available")
        return
    
    print("Monitoring GPU usage...")
    print("Press Ctrl+C to stop\n")
    
    try:
        for _ in range(duration_seconds // interval):
            print(f"\n{'='*60}")
            print(f"Time: {time.strftime('%H:%M:%S')}")
            print(f"{'='*60}")
            
            for i in range(torch.cuda.device_count()):
                allocated = torch.cuda.memory_allocated(i) / 1e9
                reserved = torch.cuda.memory_reserved(i) / 1e9
                total = torch.cuda.get_device_properties(i).total_memory / 1e9
                
                print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
                print(f"  Allocated: {allocated:.2f} GB / {total:.2f} GB ({allocated/total*100:.1f}%)")
                print(f"  Reserved:  {reserved:.2f} GB / {total:.2f} GB ({reserved/total*100:.1f}%)")
                print(f"  Free:      {total - reserved:.2f} GB")
            
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\nMonitoring stopped")

# Uncomment to monitor for 5 minutes
# monitor_gpu(duration_seconds=300, interval=10)

## 11. Generate Stories After Training

In [ ]:
# After training completes, generate some stories
from storyteller.inference.generate import StoryGenerator

# Load best model
best_checkpoint = Path(config_dict['training']['save_dir']) / 'best_model.pt'

if best_checkpoint.exists():
    print(f"Loading best model from: {best_checkpoint}")
    checkpoint = torch.load(best_checkpoint, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✓ Best model loaded")
    
    # Create generator
    generator = StoryGenerator(model, tokenizer, device=str(device))
    
    # Generate sample stories
    prompts = [
        "Once upon a time in a magical forest",
        "The brave knight embarked on a quest",
        "In a distant galaxy far away",
    ]
    
    print("\n" + "=" * 60)
    print("GENERATED STORIES")
    print("=" * 60)
    
    for i, prompt in enumerate(prompts, 1):
        print(f"\n\nStory {i}: {prompt}")
        print("-" * 60)
        
        stories = generator.generate(
            prompt=prompt,
            max_new_tokens=config_dict['generation']['max_new_tokens'],
            temperature=config_dict['generation']['temperature'],
            top_k=config_dict['generation']['top_k'],
            top_p=config_dict['generation']['top_p'],
            repetition_penalty=config_dict['generation']['repetition_penalty'],
        )
        
        print(stories[0])
else:
    print(f"Best model not found at: {best_checkpoint}")
    print("Training may not have completed yet.")